In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith
import json
from pathlib import Path

import pandas as pd

/mnt/sdd1/atharvas/formulacode/datasmith


For FC-verified

In [2]:
rem_tasks = set(
    map(
        tuple,
        pd.read_csv("dataset/pending_manual_review_tasks.csv")[["repo_name", "pr_base_sha"]].itertuples(index=False),
    )
)

all_tasks = set([
    (path.parent.name, path.name)
    for path in Path("dataset/formulacode_verified").glob("**/*")
    if path.is_dir()
    if "cache" not in str(path) and path.parents[1].name == "formulacode_verified"
])

tasks_path = Path("dataset/formulacode_verified")
tasks_path.exists(), len(rem_tasks), len(all_tasks)

(True, 48, 186)

In [3]:
all_tasks = set([
    (path.parent.name, path.name)
    for path in Path("dataset/formulacode_verified_new").glob("**/*")
    if path.is_dir()
    if "cache" not in str(path) and path.parents[1].name == "formulacode_verified_new"
])

rem_tasks = all_tasks
tasks_path = Path("dataset/formulacode_verified_new")
tasks_path.exists(), len(all_tasks)

(True, 21)

In [31]:
def get_status(task_path: Path) -> dict:
    verification_file = task_path / "verification_success.json"
    failure_file = task_path / "failure.json"
    # read the one that was written more recently
    # if verification_file.exists() and failure_file.exists():
    #     if verification_file.stat().st_mtime > failure_file.stat().st_mtime:
    #         # verification is more recent
    #         # remove failure file
    #         failure_file.unlink()
    #         d = json.loads(verification_file.read_text())
    #         d["task_path"] = str(task_path)
    #         d["ok"] = True
    #         return d
    #     else:
    #         d = json.loads(failure_file.read_text())
    #         d["task_path"] = str(task_path)
    #         d["ok"] = False
    #         return d
    if verification_file.exists():
        d = json.loads(verification_file.read_text())
        d["task_path"] = str(task_path)
        d["ok"] = True
        return d
    if failure_file.exists():
        d = json.loads(failure_file.read_text())
        d["task_path"] = str(task_path)
        d["ok"] = False
        return d
    raise FileNotFoundError(f"No status file found in {task_path}")


statuses = []
# for _, row in rem_tasks.iterrows():
# for (repo_name, pr_base_sha) in all_tasks:
for repo_name, pr_base_sha in rem_tasks:
    norm_repo = str(repo_name).replace("/", "_")
    task_path = tasks_path / norm_repo / str(pr_base_sha)
    status = get_status(task_path)
    statuses.append(status)
status_df = pd.DataFrame(statuses)
status_df["ok"].value_counts()

ok
True     17
False     4
Name: count, dtype: int64

In [32]:
status_df[~status_df["ok"]]["return_code"].value_counts()

return_code
1.0    1
Name: count, dtype: int64

In [33]:
tasks = sorted(status_df[~status_df["ok"]]["task_path"].values)
tasks

['dataset/formulacode_verified_new/pandas-dev_pandas/9ff14a3ec4259b04b25aa9ce5b23185574c2c771',
 'dataset/formulacode_verified_new/scikit-learn_scikit-learn/451f2121eab19dd25c337192a89d8c1582664d89',
 'dataset/formulacode_verified_new/scikit-learn_scikit-learn/548fc6fb680acbd26ed5213a1e7ace59722c0641',
 'dataset/formulacode_verified_new/scikit-learn_scikit-learn/91d5640a82dd0bd35c8c438ef22ff3c2cd56bce3']

In [16]:
tasks = sorted(status_df[~status_df["ok"]]["task_path"].values)
tasks

['dataset/formulacode_verified_new/pandas-dev_pandas/2b568ce82618c5e6dd2fc6f6f1d22e84c4883546',
 'dataset/formulacode_verified_new/pandas-dev_pandas/53525ea1c333579ee612244ddea4958d900844fc',
 'dataset/formulacode_verified_new/pandas-dev_pandas/9ff14a3ec4259b04b25aa9ce5b23185574c2c771',
 'dataset/formulacode_verified_new/pybamm-team_PyBaMM/dce8938907ec69892c5600efddca8589280d8a5d',
 'dataset/formulacode_verified_new/scikit-learn_scikit-learn/451f2121eab19dd25c337192a89d8c1582664d89',
 'dataset/formulacode_verified_new/scikit-learn_scikit-learn/548fc6fb680acbd26ed5213a1e7ace59722c0641',
 'dataset/formulacode_verified_new/scikit-learn_scikit-learn/91d5640a82dd0bd35c8c438ef22ff3c2cd56bce3']

In [ ]:
prompt = """
Read the instructions in AGENTS.md and then triage and fix the issue in '{task_dir}/failure.json'. Once you identify the issue, fix it, rerun verification, look at the new failure.json, and iterate until it succeeds.""".strip()

for t in tasks:
    task_dir = t.replace("/mnt/sdd1/atharvas/formulacode/datasmith/", "")
    print(prompt.format(task_dir=task_dir))
    print()
    print()


Read the instructions in AGENTS.md and then triage and fix the issue in '/mnt/sdd1/atharvas/formulacode/datasmith/dataset/formulacode_verified_new/pandas-dev_pandas/2b568ce82618c5e6dd2fc6f6f1d22e84c4883546/failure.json'. Once you identify the issue, fix it, rerun verification, look at the new failure.json, and iterate until it succeeds.



Read the instructions in AGENTS.md and then triage and fix the issue in '/mnt/sdd1/atharvas/formulacode/datasmith/dataset/formulacode_verified_new/pandas-dev_pandas/53525ea1c333579ee612244ddea4958d900844fc/failure.json'. Once you identify the issue, fix it, rerun verification, look at the new failure.json, and iterate until it succeeds.



Read the instructions in AGENTS.md and then triage and fix the issue in '/mnt/sdd1/atharvas/formulacode/datasmith/dataset/formulacode_verified_new/pandas-dev_pandas/9ff14a3ec4259b04b25aa9ce5b23185574c2c771/failure.json'. Once you identify the issue, fix it, rerun verification, look at the new failure.json, and ite

In [85]:
tasks_todo = set(status_df[~status_df["ok"]]["task_path"].values)

In [79]:
!ls dataset/formulacode_verified/uxarray_uxarray/e88b1da5257c0ae4d74b3fd5cbb16bb58215ab27

ls: cannot access 'dataset/formulacode_verified/uxarray_uxarray/e88b1da5257c0ae4d74b3fd5cbb16bb58215ab27': No such file or directory


script to dedup all_verification_successes.jsonl


In [13]:
success_jsonl = Path("dataset/all_verification_successes.jsonl")
dedup_lines = set()
for line in open("dataset/all_verification_successes.jsonl"):
    dedup_lines.add(line)


success_jsonl.write_text("".join(sorted(dedup_lines)))

10446

## Merge and make the final formulacode_verified artifacts.

Required:
 - master.parquet.
 - valid_tasks_100.json

In [2]:
def get_sha_from_dict(d):
    return set(map(lambda k: eval(k)[1], d.keys()))


def get_sha_from_image(ecr_image: str) -> str:
    # 204464138089.dkr.ecr.us-east-1.amazonaws.com/formulacode/all:pywavelets-pywt-577e5a9b3a2eb228539fd3e69301edc603b845df--final'
    # to '577e5a9b3a2eb228539fd3e69301edc603b845df'
    # OR
    # docker.io/formulacode/all:tiledb-inc-tiledb-py-28714d9b25d44d6c6c1f318525184d3784b7de00--final
    # to '28714d9b25d44d6c6c1f318525184d3784b7de00'
    img_name = ecr_image.split(":")[-1].split("--")[0]
    sha = img_name.split("-")[-1]
    return sha


def get_sha_from_entry(entry: dict) -> str:
    if "ecr_image" in entry:
        return get_sha_from_image(entry["ecr_image"])
    elif "dockerhub_image" in entry:
        return get_sha_from_image(entry["dockerhub_image"])
    raise ValueError("No image found in entry")

In [3]:
master_dataset = pd.read_parquet("scratch/artifacts/pipeflush/perfonly_commits_master.parquet")
# master_dataset = pd.read_parquet("scratch/artifacts/pipeflush/perfonly_commits_with_patch_valid_tasks_200_finished.parquet")

successful_verification_files = [
    Path("dataset/all_verification_successes.jsonl"),
    Path("dataset/all_verification_successes.bak.jsonl"),
]

task_speedups_files = [
    Path("/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/valid_tasks_new_200.json"),
    Path("/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/valid_tasks_200.json"),
]

verified_shas = {
    get_sha_from_entry(json.loads(line))
    for verification_file in successful_verification_files
    for line in verification_file.read_text().splitlines()
}

task_speedups = {
    eval(k): v
    for task_speedups_file in task_speedups_files
    for k, v in json.loads(task_speedups_file.read_text()).items()
}

In [4]:
# subset of master_dataset where sha is in verified_shas
verified_master_dataset = master_dataset[master_dataset["pr_base_sha"].isin(verified_shas)].copy()
verified_master_dataset["max_speedup"] = pd.MultiIndex.from_frame(
    verified_master_dataset[["repo_name", "pr_merge_commit_sha"]]
).map(pd.Series(task_speedups))

verified_master_dataset = verified_master_dataset[
    (verified_master_dataset["max_speedup"] < 500) & (verified_master_dataset["max_speedup"] > 1.33)
]
verified_master_dataset = verified_master_dataset.sort_values(["repo_name", "pr_merge_commit_sha"])
verified_master_dataset.shape

(108, 162)

In [ ]:
verified_master_dataset[5]

In [ ]:
verified_master_dataset.to_parquet("scratch/artifacts/pipeflush/perfonly_commits_master_verified.parquet", index=False)

Path("dataset/valid_tasks.json").write_text(
    verified_master_dataset.groupby(["repo_name", "pr_merge_commit_sha"])["max_speedup"]
    .max()
    # .reset_index(name="max_speedup")
    .to_json(orient="index")
)

In [10]:
# make 9 equal sized splits of verified_master_dataset.
num_splits = 9
split_size = len(verified_master_dataset) // num_splits

In [ ]:
split_dataframes = [verified_master_dataset.iloc[i * split_size : (i + 1) * split_size] for i in range(num_splits)]

for i in range(num_splits):
    split_df = verified_master_dataset.iloc[i * split_size : (i + 1) * split_size]
    Path(f"dataset/valid_tasks_split_{i}.json").write_text(
        split_df.groupby(["repo_name", "pr_merge_commit_sha"])["max_speedup"]
        .max()
        # .reset_index(name="max_speedup")
        .to_json(orient="index")
    )

In [17]:
cxx_error_pandas = set(
    filter(
        lambda s: s.startswith("#") == False,
        Path("dataset/formulacode_verified/pandas-dev_pandas/cxxabi_1_3_15_tasks.txt").read_text().splitlines(),
    )
)

error_dataset = verified_master_dataset[verified_master_dataset["pr_base_sha"].isin(cxx_error_pandas)]

In [11]:
i = 5
verified_master_dataset.iloc[i * split_size : (i + 1) * split_size].iloc[5]["pr_base_sha"]

'6d89d8c900ea27fe6c55f204d6b96961e73aa67f'

In [18]:
# which of the splits are 100% error tasks?
for i in range(num_splits):
    split_df = verified_master_dataset.iloc[i * split_size : (i + 1) * split_size]
    num_error_tasks = split_df[split_df["pr_base_sha"].isin(cxx_error_pandas)].shape[0]
    total_tasks = split_df.shape[0]
    print(f"Split {i}: {num_error_tasks}/{total_tasks} error tasks ({(num_error_tasks / total_tasks) * 100:.2f}%)")

Split 0: 0/12 error tasks (0.00%)
Split 1: 0/12 error tasks (0.00%)
Split 2: 2/12 error tasks (16.67%)
Split 3: 4/12 error tasks (33.33%)
Split 4: 4/12 error tasks (33.33%)
Split 5: 3/12 error tasks (25.00%)
Split 6: 2/12 error tasks (16.67%)
Split 7: 3/12 error tasks (25.00%)
Split 8: 0/12 error tasks (0.00%)


In [19]:
# Make a new split that contains only the error tasks (I've fixed them)
error_verified_master_dataset = verified_master_dataset[verified_master_dataset["pr_base_sha"].isin(cxx_error_pandas)]

In [20]:
Path("dataset/valid_tasks_remaining.json").write_text(
    error_verified_master_dataset.groupby(["repo_name", "pr_merge_commit_sha"])["max_speedup"]
    .max()
    # .reset_index(name="max_speedup")
    .to_json(orient="index")
)

1478